# GOSS: Gradient-based One-Side Sampling in LightGBM

This is a clever optimization technique that LightGBM uses to accelerate training while maintaining model quality. Let me break down the mechanics and intuition.

## Core Idea

GOSS exploits the observation that **instances with large gradients contribute more to information gain** (split quality) than instances with small gradients. So instead of using all data for each split decision, GOSS:

1. **Keeps all instances with large gradients** (top `a%` by absolute gradient magnitude)
2. **Randomly samples instances with small gradients** (bottom `(1-a)%`, keeping `b%` of those)
3. Uses this subset to find the best split

This reduces the data size per iteration while preserving the most informative samples.

## Why This Works

### Gradient as Importance Signal

In tree boosting, the gradient $g_i$ for instance $i$ represents how "wrong" the current model is on that instance:
- **Large $|g_i|$** → instance is hard to fit, contains lots of signal
- **Small $|g_i|$** → instance is easy to fit, prediction is already close

When finding a split, the gain depends on how well the split separates instances by their gradients. Instances with large gradients have more influence on this gain calculation.

### Mathematical Justification

The information gain for a split is roughly:
$$\text{Gain} = \frac{(\sum_{L} g_i)^2}{n_L} + \frac{(\sum_{R} g_i)^2}{n_R} - \frac{(\sum g_i)^2}{n}$$

The large-gradient instances dominate these sums, so removing small-gradient instances doesn't hurt split quality much—but it dramatically speeds up histogram construction.

## Practical Parameters

LightGBM uses two hyperparameters:

- **`top_rate` (a)**: Percentage of top-gradient instances to keep (default ~0.2 or 20%)
- **`other_rate` (b)**: Sampling rate for remaining instances (default ~0.1 or 10%)

So with defaults:
- Keep top 20% by gradient
- Randomly sample 10% of the remaining 80%
- Total data used ≈ 20% + 10% × 80% = **28% of original data**

## Speed vs. Accuracy Trade-off

| Setting | Training Speed | Model Quality | Use Case |
|---------|---|---|---|
| `top_rate=0.2, other_rate=0.1` | ~3–4× faster | ~99% accuracy vs. full data | Imbalanced classification (fraud, clicks) |
| `top_rate=0.1, other_rate=0.05` | ~5–6× faster | ~97–98% accuracy vs. full data | Very large datasets (100M+ rows) |
| `top_rate=1.0, other_rate=1.0` | Baseline | Baseline (no GOSS) | High-accuracy requirements |

## In Your SentinelPay Context

GOSS is **particularly valuable** for fraud detection:

1. **Fraud is imbalanced**: Most transactions are legitimate. Large-gradient instances (hard cases, actual fraud, borderline legit) get priority; easy negatives are undersampled.

2. **Preserves precision**: By focusing on informative samples, GOSS doesn't hurt PR-AUC or Precision-Recall curves—the metrics that matter for fraud.

3. **Training speed**: For your ~$4B PayFac portfolio analysis, GOSS can make training 3–6× faster without sacrificing the AUC improvements you've achieved.

## Enabling GOSS in LightGBM

```python
import lightgbm as lgb
model = lgb.LGBMClassifier(objective='binary',
                           boosting_type='gbdt',
                           metric='auc',
                           learning_rate=0.05,
                           num_leaves=31,
                           max_depth=5,
                            top_rate =  0.2,           # Keep top 20% by gradient
                            other_rate = 0.1,         # Sample 10% of rest
                            # Other params...)
}

model = lgb.train(params, train_data, num_boost_round=100)
```